# 01c — Saraga Carnatic: Exploratory Data Analysis

This notebook characterises the processed Carnatic subset (60 transcribed MIDI files).
Carnatic music features a distinct melodic grammar (gamaka — continuous pitch inflections,
oscillations) and rhythmic structure (tala cycles with subdivisions called laghu, drutam,
anudrutam) that standard REMI tokenisation cannot fully represent.

**Prerequisite:** Run `00c_carnatic_prep.ipynb` first.


In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.midi_utils import (
    load_midi, analyse_midi, get_pitch_class_histogram,
    pitch_class_entropy, piano_roll_plot
)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

PC_LABELS = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [2]:
def build_stats_df(midi_dir, meta_df=None, id_col=None):
    """Analyse all MIDI files in midi_dir; optionally merge metadata."""
    midi_files = sorted(midi_dir.glob("*.mid")) + sorted(midi_dir.glob("*.midi"))
    print(f"Analysing {len(midi_files)} MIDI files ...")
    records = [analyse_midi(p) for p in midi_files]
    df = pd.DataFrame(records)
    if "error" in df.columns:
        bad = df["error"].notna().sum()
        if bad:
            print(f"  Warning: {bad} files failed to load.")
        df = df[df["error"].isna()].drop(columns=["error"])
    df["filename"] = [Path(p).name for p in df["path"]]
    if meta_df is not None and id_col is not None:
        df = df.merge(meta_df, left_on="filename", right_on=id_col, how="left")
    return df

In [3]:
def summary_panel(df, tradition_name, save_path):
    """4-panel summary figure: duration, note density, pitch range, PC entropy."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f"{tradition_name} — EDA Summary (n={len(df)})", fontsize=13, y=1.01)

    # Duration
    ax = axes[0, 0]
    df["duration_s"].div(60).plot.hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
    ax.axvline(df["duration_s"].mean()/60, color="red", linestyle="--", label=f"mean={df['duration_s'].mean()/60:.1f} min")
    ax.set_xlabel("Duration (minutes)")
    ax.set_title("Duration Distribution")
    ax.legend(fontsize=8)

    # Note density
    ax = axes[0, 1]
    df["note_density"].plot.hist(bins=25, ax=ax, color="seagreen", edgecolor="white")
    ax.axvline(df["note_density"].mean(), color="red", linestyle="--", label=f"mean={df['note_density'].mean():.2f}")
    ax.set_xlabel("Notes per second")
    ax.set_title("Note Density Distribution")
    ax.legend(fontsize=8)

    # Pitch range
    ax = axes[1, 0]
    df["pitch_range"].plot.hist(bins=25, ax=ax, color="darkorange", edgecolor="white")
    ax.axvline(df["pitch_range"].mean(), color="red", linestyle="--", label=f"mean={df['pitch_range'].mean():.1f}")
    ax.set_xlabel("Pitch range (semitones)")
    ax.set_title("Pitch Range Distribution")
    ax.legend(fontsize=8)

    # PC entropy
    ax = axes[1, 1]
    df["pc_entropy"].plot.hist(bins=25, ax=ax, color="mediumpurple", edgecolor="white")
    ax.axvline(df["pc_entropy"].mean(), color="red", linestyle="--", label=f"mean={df['pc_entropy'].mean():.3f}")
    ax.set_xlabel("Pitch Class Entropy (bits)")
    ax.set_title("PC Entropy Distribution\n(Yang & Lerch, 2020)")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")


def mean_pc_histogram_plot(df_midi_paths, tradition_name, save_path):
    """Plot the mean pitch class histogram across all pieces."""
    hists = []
    for p in df_midi_paths:
        pm = load_midi(p)
        if pm:
            hists.append(get_pitch_class_histogram(pm))
    if not hists:
        return
    mean_hist = np.mean(hists, axis=0)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(PC_LABELS, mean_hist, color="steelblue", edgecolor="white")
    ax.set_xlabel("Pitch class")
    ax.set_ylabel("Mean relative frequency")
    ax.set_title(f"{tradition_name} — Mean Pitch Class Histogram (n={len(hists)})")
    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")

In [4]:
MIDI_DIR = PROJECT_ROOT / "data" / "processed" / "carnatic" / "midi"
META_CSV = PROJECT_ROOT / "data" / "metadata" / "carnatic_tracks.csv"

midi_files = list(MIDI_DIR.glob("*.mid"))
if not midi_files:
    raise FileNotFoundError(
        "No MIDI files in data/processed/carnatic/midi/. "
        "Run notebook 00c_carnatic_prep.ipynb first."
    )
print(f"Found {len(midi_files)} MIDI files.")

meta = pd.read_csv(META_CSV)
print(f"Metadata rows: {len(meta)}")
meta[["title", "raga", "taal", "metadata_source", "concert_folder"]].head(5)

Found 60 MIDI files.
Metadata rows: 60


,title,raga,taal,metadata_source,concert_folder
0,Shloka Sri Ramachandra Shrita Parijata,Shloka Sri Ramachandra Shrita Parijata,NaN,title_fallback,A V K Rajasimhan at Arkay by Angarai V K Rajas...
1,Nera Nammiti,Nera Nammiti,NaN,title_fallback,A V K Rajasimhan at Arkay by Angarai V K Rajas...
2,Siddhi Vinayakam,shanmukhapriya,rupaka,json,Akkarai Sisters at Arkay by Akkarai Sisters
3,Manadirkkugandhadu,sindhubhairavi,adi,json,Akkarai Sisters at Arkay by Akkarai Sisters
4,Karunimpa Idi,sahana,adi,json,Akkarai Sisters at Arkay by Akkarai Sisters


## 1. Metadata quality

In [5]:
print("Metadata source breakdown:")
print(meta["metadata_source"].value_counts().to_string())
print(f"\nUnique ragas (incl. transliterated): {meta['raga'].nunique()}")
print(f"Concert folders (performers)         : {meta['concert_folder'].nunique()}")
print("\nNote: many raga names are Unicode transliterations from the JSON 'name' field.")
print("Where raaga array was empty, the piece title was used as raga proxy.")

Metadata source breakdown:
metadata_source
json              36
title_fallback    24

Unique ragas (incl. transliterated): 56
Concert folders (performers)         : 17

Note: many raga names are Unicode transliterations from the JSON 'name' field.
Where raaga array was empty, the piece title was used as raga proxy.


## 2. Compute statistics

In [6]:
stats = build_stats_df(MIDI_DIR)
print(f"Files analysed: {len(stats)}")
print("\nDescriptive statistics:")
print(stats[["duration_s", "note_count", "note_density", "pitch_range", "pc_entropy"]].describe().round(3))

Analysing 60 MIDI files ...
Files analysed: 60

Descriptive statistics:
       duration_s  note_count  note_density  pitch_range  pc_entropy
count      60.000      60.000        60.000       60.000      60.000
mean      921.380    2801.333         3.319       53.433       2.791
std       868.446    2484.001         1.009       12.071       0.340
min        27.350      99.000         1.370       27.000       2.035
25%       291.165    1028.000         2.468       44.000       2.603
50%       470.780    1617.500         3.466       52.500       2.771
75%      1437.745    4127.000         4.031       63.250       3.053
max      3584.050   10696.000         5.244       74.000       3.354


## 3. Distribution plots

In [7]:
summary_panel(stats, "Carnatic Classical (Saraga)", RESULTS_DIR / "eda_carnatic_summary.png")

Saved → results/eda_carnatic_summary.png


## 4. Mean pitch class histogram

In [8]:
mean_pc_histogram_plot(stats["path"].tolist(), "Carnatic Classical (Saraga)",
                       RESULTS_DIR / "eda_carnatic_pc_histogram.png")

Saved → results/eda_carnatic_pc_histogram.png


## 5. Performer (concert) distribution

Carnatic tracks are drawn from 27 concert recordings. The distribution below shows
how many tracks each performer contributes to the selected subset.


In [9]:
perf_counts = meta["concert_folder"].value_counts()
fig, ax = plt.subplots(figsize=(14, 5))
perf_counts.plot(kind="bar", ax=ax, color="teal", edgecolor="white")
ax.set_xlabel("Concert / Performer")
ax.set_ylabel("Tracks")
ax.set_title(f"Carnatic — Tracks per Performer (n={len(meta)})")
ax.tick_params(axis="x", rotation=60, labelsize=7)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "eda_carnatic_performer_dist.png"), dpi=150)
plt.show()

## 6. Sample piano roll

In [10]:
sample_path = stats["path"].iloc[0]
pm = load_midi(sample_path)
sample_name = Path(sample_path).stem[:60]

fig, ax = plt.subplots(figsize=(14, 4))
piano_roll_plot(pm, ax, time_start=30, time_end=60,
                title=f"Piano roll sample (30–60 s) — {sample_name}")
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "eda_carnatic_piano_roll.png"), dpi=150)
plt.show()

## 7. Save summary statistics

In [11]:
stats["tradition"] = "carnatic"
stats.to_csv(RESULTS_DIR / "eda_carnatic_stats.csv", index=False)
print("Saved → results/eda_carnatic_stats.csv")
print(f"\nPC entropy mean : {stats['pc_entropy'].mean():.3f} bits")
print(f"Note density mean: {stats['note_density'].mean():.2f} notes/s")

Saved → results/eda_carnatic_stats.csv

PC entropy mean : 2.791 bits
Note density mean: 3.32 notes/s
